# Grid Foundation Model (GENCO)–Embedded Optimization for AC-Feasible Unit Commitment

**Waquar Kaleem · The Pennsylvania State University**

*Research prototype*

IEEE 57 · four hourly periods · one switchable generator · CPU

For period $t$, GENCO predicts generation and voltages from the online
indicators $u_t^{on}$ and model inputs $x_t^{in}$:

$$x_t^{out}=f_t^{FM}(u_t^{on},x_t^{in};\beta),\qquad
\widehat x_t^{el}=A_t(x_t^{out},u_t^{on}).$$

Here $\beta$ contains the trained parameters. The function $A_t$ allocates
bus reactive generation among online generators and forms the free electrical
coordinates of generation and voltages, with powers in per unit and angles
in radians.

For fixed $u_t^{on}$, the correction locally solves

$$\min_{x_t^{el}\in\mathcal C_t(u_t^{on})}
\frac12\|x_t^{el}-\widehat x_t^{el}\|_2^2,\qquad
x_t^{OPF}=h_t^{el}(x_t^{el};u_t^{on}).$$

$\mathcal C_t$ enforces the original AC equations and operating limits;
$h_t^{el}$ restores fixed coordinates and computes voltage products and line
flows. We optimize commitment and model inputs jointly while keeping the
physical network, demand and costs fixed. The implementation combines
**GENCO inference → sparse KKT correction → implicit differentiation at regular KKT points**
with **UC coordination in SCIP and multiplier/penalty updates**. The complete
schedule must satisfy AC equations, operating limits, commitment logic and ramps.


## 1. GENCO parameters

In [ ]:
MODEL_SIZE = "base"  # "small" or "base"
SEED = 0  # 0 or 1


## 2. Setup

The notebook includes the source and case. It downloads the matching GENCO checkpoint and normalization data automatically. Automatic installation requires Python 3.11 on Linux x86_64.

In [ ]:
# Included source archive; expand this cell to inspect the extraction code.
import base64, hashlib, io, json, os, pathlib, subprocess, sys, zipfile

WORK = pathlib.Path.cwd() / "gfmo_demo"
payload = base64.b64decode(
    "..."
)
assert (
    hashlib.sha256(payload).hexdigest()
    == "b73a9d5ef56ede0362c6a73154740dab10cc35c71dcd1ec1424e090bf56ee949"
)
with zipfile.ZipFile(io.BytesIO(payload)) as archive:
    for item in archive.infolist():
        relative = pathlib.PurePosixPath(item.filename)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError("Unsafe source path")
        target = WORK / str(relative)
        data = archive.read(item)
        if target.exists() and target.read_bytes() != data:
            raise ValueError(
                f"Existing source differs: {target}. Use a fresh work directory."
            )
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(data)
print("Source and case ready.")


In [ ]:
import importlib.util
from IPython.display import display, ProgressBar, SVG

spec = importlib.util.spec_from_file_location("gfmo_setup", WORK / "setup_demo.py")
setup = importlib.util.module_from_spec(spec)
spec.loader.exec_module(setup)
progress = ProgressBar(3)
display(progress)
setup_status = display("Preparing CPU environment…", display_id=True)


def advance(step, label):
    progress.progress = step
    setup_status.update(label)


PYTHON = setup.environment(progress=advance)
setup_status.update("Checking GENCO weights, normalization data and network…")
with (WORK / "setup.log").open("a") as log:
    subprocess.run(
        [
            PYTHON,
            str(WORK / "setup_demo.py"),
            "--size",
            MODEL_SIZE,
            "--seed",
            str(SEED),
        ],
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
    subprocess.run(
        [
            PYTHON,
            "-c",
            "import sys; sys.path.insert(0,'src'); import opf_adapter, corrected_period, manuscript_decomposition",
        ],
        cwd=WORK,
        env=dict(os.environ, MPLBACKEND="Agg"),
        check=True,
        stdout=log,
        stderr=subprocess.STDOUT,
    )
progress.progress = 3
setup_status.update("Setup complete")
print("GENCO", MODEL_SIZE, "· seed", SEED, "· IEEE 57 ready")


## 3. Solve

A fresh run has a 300-second search budget. The bar shows elapsed time, not convergence. Installation and downloads are outside the clock.

In [ ]:
import time, signal, uuid

budget = 300
elapsed = ProgressBar(budget)
display(elapsed)
solve_status = display(
    "Loading GENCO and initializing the UC schedule…", display_id=True
)
live_plot = display("The first feasible schedule will appear here.", display_id=True)
last_plot = None
marker = WORK / (".progress_" + uuid.uuid4().hex + ".json")
run_env = dict(
    os.environ,
    GFMO_DEMO_PROGRESS_FILE=str(marker),
    MPLBACKEND="Agg",
    MPLCONFIGDIR=str(WORK / ".cache/matplotlib"),
    XDG_CACHE_HOME=str(WORK / ".cache"),
)
started = time.monotonic()
with (WORK / "notebook_solve.log").open("w") as log:
    process = subprocess.Popen(
        [
            PYTHON,
            str(WORK / "run_demo.py"),
            "--size",
            MODEL_SIZE,
            "--seed",
            str(SEED),
            "--seconds",
            str(budget),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        start_new_session=True,
        env=run_env,
    )
    try:
        while process.poll() is None:
            elapsed.progress = min(budget, int(time.monotonic() - started))
            if marker.exists():
                try:
                    current = pathlib.Path(json.loads(marker.read_text())["folder"])
                    report = json.loads((current / "solver.json").read_text())
                    completed = sum(
                        c.get("phase") == "initialization"
                        and c.get("status") == "accepted corrected feasibility point"
                        for c in report.get("calls", [])
                    )
                    phase = (
                        f"GENCO + AC correction · {completed}/4 periods initialized"
                        if report["status"] == "initializing"
                        else f"UC coordination + input optimization · {len(report.get('history', []))} iterations"
                    )
                    best = report.get("best")
                    feasible = (
                        "no feasible schedule yet"
                        if best is None
                        else f"best cost {best['objective']:,.2f}"
                    )
                    solve_status.update(
                        f"{int(time.monotonic()-started)} s · {phase} · {feasible}"
                    )
                    if best is not None and best["accepted_at_s"] != last_plot:
                        figure = current / "progress.svg"
                        plot = subprocess.run(
                            [
                                PYTHON,
                                str(WORK / "demo_view.py"),
                                str(current / "solver.json"),
                                str(figure),
                            ],
                            env=run_env,
                            stdout=log,
                            stderr=subprocess.STDOUT,
                        )
                        if plot.returncode == 0:
                            live_plot.update(SVG(filename=str(figure)))
                            last_plot = best["accepted_at_s"]
                except (FileNotFoundError, json.JSONDecodeError):
                    pass
            time.sleep(1)
    except KeyboardInterrupt:
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            pass
        try:
            os.killpg(process.pid, signal.SIGKILL)
        except ProcessLookupError:
            pass
        process.wait()
        raise
if process.returncode:
    raise RuntimeError((WORK / "notebook_solve.log").read_text()[-4000:])
run_folder = pathlib.Path(json.loads((WORK / "latest_run.json").read_text())["folder"])
summary = json.loads((run_folder / "summary.json").read_text())
solve_status.update("Solve complete · checking the full UC–ACOPF schedule")
print("AC-feasible UC schedule:", summary["accepted"])
print("Time to first feasible schedule (s):", summary["first_feasible_s"])
print("Total computation time (s):", round(summary["process_elapsed_s"], 2))
print("Operating cost:", summary.get("objective", "—"))


## 4. Check the schedule

The plots show the current best schedule. The residual comparison evaluates
GENCO predictions at the selected input values and their corrected outputs
against the original physical network.

The checker recomputes all original UC–ACOPF constraints. Tolerances:
$10^{-3}$ MW/MVAr/MVA, $10^{-5}$ p.u., $10^{-6}$ rad; exact binary decisions.


In [ ]:
if summary["accepted"]:
    witness = json.loads((run_folder / "witness.json").read_text())
    print("Online indicators (rows: periods; columns: generators)")
    for row in witness["u"]:
        print(" ".join(str(int(v)) for v in row))
    print("\nMaximum constraint violations")
    for name, value in summary["physical_check"]["max_violations"].items():
        print(f"{name}: {value:.3e}")
else:
    print(
        "No feasible schedule found within the budget. See solver.log and solver.json."
    )


[GENCO models](https://huggingface.co/gridfm/genco-opf-datakit) · [IEEE57 data](https://huggingface.co/datasets/gridfm/opf_small_case57_ieee) · [Graphkit](https://github.com/gridfm/gridfm-graphkit)

Constructed demonstration on source scenario 308; not held-out evaluation.
Local correction does not guarantee success or global optimality. The target
model includes nonnegative reactive generation and the principal angle branch.
Original code: evaluation-only license. Third-party terms are preserved.
Contact: **wzk5140@psu.edu**. AI-assisted development and documentation.
